# Review Aluminium confirmed raw spots

This notebook renders every confirmed spot in `confirmed_spots_merged.h5` for `Al`, `Al_big_grains`, and `Al_small_grains`. The crop comes from the saved bbox and the grayscale image is the raw detector frame; the red contour is the saved mask. Odd-looking crops are the ones to redo in `spot_browser.ipynb`.

In [ ]:
from pathlib import Path
import math

import h5py
import hdf5plugin  # registers ESRF compression filters
import matplotlib.pyplot as plt
import numpy as np


def find_project_dir():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "train.py").exists() and (candidate / "augment_data").exists():
            return candidate
    raise RuntimeError("Could not find project directory containing train.py and augment_data/.")


PROJECT_DIR = find_project_dir()
DATA_ESRF = PROJECT_DIR.parent / "data_esrf"
CATALOG_FILE = DATA_ESRF / "confirmed_spots_merged.h5"
RAW_ROOTS = [DATA_ESRF]
RAW_KEY = "instrument/detector_0/data"
AL_SCANS = ("Al", "Al_big_grains", "Al_small_grains")
OUT_DIR = PROJECT_DIR / "prediction_previews" / "confirmed_al_raw_spots"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project dir : {PROJECT_DIR}")
print(f"Catalog     : {CATALOG_FILE}  exists={CATALOG_FILE.exists()}")
print(f"Output dir  : {OUT_DIR}")

In [ ]:
def dataset_exists(h5_file, key):
    try:
        with h5py.File(h5_file, "r") as f:
            return key in f and isinstance(f[key], h5py.Dataset)
    except OSError:
        return False


def first_3d_dataset(h5_file):
    found = []
    try:
        with h5py.File(h5_file, "r") as f:
            def visitor(name, obj):
                if isinstance(obj, h5py.Dataset) and obj.ndim >= 3:
                    found.append(name)
            f.visititems(visitor)
    except OSError as exc:
        print(f"Skipping unreadable HDF5 candidate {h5_file}: {exc}")
        return None
    return found[0] if found else None


def find_scan_dir(scan):
    for root in RAW_ROOTS:
        scan_dir = root / scan
        if scan_dir.exists():
            return scan_dir
    raise FileNotFoundError(f"Could not find scan directory for {scan!r} in {RAW_ROOTS}")


def find_raw_file(scan):
    scan_dir = find_scan_dir(scan)
    h5_files = sorted(
        p for p in scan_dir.glob("*.h5")
        if p.name != "segvol.h5" and "dark" not in p.name.lower() and "confirmed" not in p.name.lower()
    )
    if not h5_files:
        raise FileNotFoundError(f"No raw .h5 file found in {scan_dir}")

    for h5_file in h5_files:
        if dataset_exists(h5_file, RAW_KEY):
            return h5_file, RAW_KEY

    for h5_file in h5_files:
        key = first_3d_dataset(h5_file)
        if key is not None:
            print(f"Using fallback raw dataset {key!r} in {h5_file.name} for scan {scan}")
            return h5_file, key

    raise FileNotFoundError(f"No 3D raw dataset found in {scan_dir}")


RAW_CACHE = {}


def raw_source(scan):
    if scan not in RAW_CACHE:
        RAW_CACHE[scan] = find_raw_file(scan)
    return RAW_CACHE[scan]


def robust_limits(array, lower=1, upper=99.8):
    values = np.asarray(array)[np.isfinite(array)]
    values = values[values > 0]
    if values.size == 0:
        return 0.0, 1.0
    vmin, vmax = np.percentile(values, [lower, upper])
    if vmax <= vmin:
        vmax = float(values.max()) if values.max() > 0 else 1.0
        vmin = 0.0
    return float(vmin), float(vmax)


def collect_al_spots(catalog_file=CATALOG_FILE, scans=AL_SCANS):
    spots = []
    with h5py.File(catalog_file, "r") as f:
        for scan in scans:
            if scan not in f:
                print(f"Missing scan in catalog: {scan}")
                continue
            for name, group in sorted(f[scan].items()):
                if not isinstance(group, h5py.Group) or "bbox" not in group or "mask" not in group:
                    continue
                bbox = np.asarray(group["bbox"][()], dtype=np.int32)
                mask = np.asarray(group["mask"][()], dtype=bool)
                frame = int(group.attrs.get("frame"))
                blob_id = int(group.attrs.get("blob_id", -1))
                spots.append({
                    "path": f"{scan}/{name}",
                    "scan": scan,
                    "frame": frame,
                    "blob_id": blob_id,
                    "bbox": bbox,
                    "mask": mask,
                    "area_px": int(mask.sum()),
                })
    return spots


def load_raw_crop(spot):
    raw_file, raw_key = raw_source(spot["scan"])
    r0, c0, r1, c1 = map(int, spot["bbox"])
    with h5py.File(raw_file, "r") as f:
        data = f[raw_key]
        frame = int(spot["frame"]) % data.shape[0]
        crop = np.asarray(data[frame, r0:r1, c0:c1], dtype=np.float32)
    return crop, raw_file, raw_key


SPOTS = collect_al_spots()
print(f"Loaded {len(SPOTS)} Aluminium spot(s)")
for scan in AL_SCANS:
    print(f"{scan:<16} {sum(s['scan'] == scan for s in SPOTS):>3}")

In [ ]:
# Write one PNG per spot.
written = []
problems = []

for spot in SPOTS:
    try:
        raw_crop, raw_file, raw_key = load_raw_crop(spot)
        mask = spot["mask"]
        vmin, vmax = robust_limits(raw_crop)

        fig, ax = plt.subplots(figsize=(4, 4), dpi=140)
        ax.imshow(raw_crop, cmap="gray", vmin=vmin, vmax=vmax, origin="upper")
        if mask.shape == raw_crop.shape and mask.any():
            ax.contour(mask, levels=[0.5], colors="red", linewidths=0.8)
        else:
            problems.append((spot["path"], f"mask shape {mask.shape} != raw crop shape {raw_crop.shape}"))
        ax.set_title(
            f"{spot['scan']} frame {spot['frame']} blob {spot['blob_id']}\n"
            f"bbox={spot['bbox'].tolist()} area={spot['area_px']}",
            fontsize=8,
        )
        ax.axis("off")
        fig.tight_layout(pad=0.2)

        filename = f"{spot['scan']}_frame_{spot['frame']:04d}_blob_{spot['blob_id']:04d}.png"
        out_path = OUT_DIR / filename
        fig.savefig(out_path, bbox_inches="tight", pad_inches=0.02)
        plt.close(fig)
        written.append(out_path)
    except Exception as exc:
        problems.append((spot["path"], repr(exc)))

print(f"Wrote {len(written)} individual PNG(s) to {OUT_DIR}")
if problems:
    print("Problems:")
    for path, problem in problems:
        print(f" - {path}: {problem}")

In [ ]:
# Contact sheet for quick visual triage.
n = len(SPOTS)
cols = 5
rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.0, rows * 3.2), squeeze=False, dpi=130)

for ax in axes.flat:
    ax.axis("off")

for ax, spot in zip(axes.flat, SPOTS):
    raw_crop, _, _ = load_raw_crop(spot)
    mask = spot["mask"]
    vmin, vmax = robust_limits(raw_crop)
    ax.imshow(raw_crop, cmap="gray", vmin=vmin, vmax=vmax, origin="upper")
    if mask.shape == raw_crop.shape and mask.any():
        ax.contour(mask, levels=[0.5], colors="red", linewidths=0.6)
    ax.set_title(f"{spot['scan']}\nf{spot['frame']} b{spot['blob_id']}", fontsize=7)
    ax.axis("off")

fig.tight_layout(pad=0.4)
sheet_path = OUT_DIR / "all_al_confirmed_raw_spots_contact_sheet.png"
fig.savefig(sheet_path, bbox_inches="tight", pad_inches=0.02)
plt.show()
print(f"Contact sheet: {sheet_path}")

In [ ]:
# Text index for matching a suspicious image back to the HDF5 group.
for spot in SPOTS:
    print(
        f"{spot['path']:<48} frame={spot['frame']:>4} "
        f"blob={spot['blob_id']:>4} bbox={spot['bbox'].tolist()} area={spot['area_px']}"
    )